# Run fMRIPrep on Echo-2 for AdView

This notebook runs **fMRIPrep on the echo-2 series** for the AdView task in the `g25` project.

Important choices:
- It uses the existing repo filter file at `code/fmriprep_config.json` so the run stays aligned with the project defaults.
- It adds `--task-id adview` and `--echo-idx 2` to restrict processing to the AdView task and the second echo.
- It references your FreeSurfer license by **file path** (`~/Desktop/license.txt`) instead of embedding the license contents in the notebook.
- It writes to a **separate output directory** so it does not overwrite the standard multi-echo fMRIPrep derivatives.

If you later feed these outputs into FEAT, use the **echo-2 preprocessed BOLD** file that this notebook finds at the end.


## Why a separate output directory?

The project scripts currently expect the usual fMRIPrep output pattern without an `echo-` label. An echo-specific run is a different preprocessing branch, so this notebook writes to `derivatives/fmriprep_echo2_adview` to keep it clearly separated from any standard fMRIPrep outputs.

That also means the repo's default confound extractor may need a small filename adjustment later if you want to use these echo-2 derivatives directly in FEAT.


In [ ]:
from __future__ import annotations

from pathlib import Path
import shlex
import subprocess

PROJECT_ROOT = Path.home() / "Downloads" / "g25"
BIDS_DIR = PROJECT_ROOT / "bids"
OUTPUT_DIR = PROJECT_ROOT / "derivatives" / "fmriprep_echo2_adview"
WORK_DIR = Path.home() / "scratch" / "g25" / "fmriprep_echo2_adview"
FILTER_FILE = PROJECT_ROOT / "code" / "fmriprep_config.json"
FS_LICENSE_FILE = Path.home() / "Desktop" / "license.txt"

SUBJECTS = ["11039"]  # add more IDs like ["11028", "11039", "11450"]
TASK_ID = "adview"
ECHO_IDX = 2
NTHREADS = 14
OMP_NTHREADS = 1
MEM_MB = 24000
RUN_REAL_FMRIPREP = False  # switch to True only when you are ready to launch the job

PROJECT_ROOT, OUTPUT_DIR, FS_LICENSE_FILE


In [ ]:
required_paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "BIDS_DIR": BIDS_DIR,
    "FILTER_FILE": FILTER_FILE,
    "FS_LICENSE_FILE": FS_LICENSE_FILE,
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required path(s): "
        + ", ".join(f"{name}={required_paths[name]}" for name in missing)
    )

missing_subjects = [sub for sub in SUBJECTS if not (BIDS_DIR / f"sub-{sub}").exists()]
if missing_subjects:
    raise FileNotFoundError(f"Missing BIDS folders for: {missing_subjects}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Preflight checks passed.")
print(f"BIDS dir: {BIDS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Work dir: {WORK_DIR}")
print(f"Filter file: {FILTER_FILE}")
print(f"FreeSurfer license: {FS_LICENSE_FILE}")


In [ ]:
cmd = [
    "fmriprep",
    str(BIDS_DIR),
    str(OUTPUT_DIR),
    "participant",
    "--participant-label",
    *SUBJECTS,
    "--task-id",
    TASK_ID,
    "--echo-idx",
    str(ECHO_IDX),
    "--stop-on-first-crash",
    "--skip-bids-validation",
    "--nthreads",
    str(NTHREADS),
    "--omp-nthreads",
    str(OMP_NTHREADS),
    "--mem-mb",
    str(MEM_MB),
    "--output-spaces",
    "MNI152NLin6Asym",
    "--bids-filter-file",
    str(FILTER_FILE),
    "--fs-no-reconall",
    "--fs-license-file",
    str(FS_LICENSE_FILE),
    "-w",
    str(WORK_DIR),
    "-v",
]

cmd_str = " ".join(shlex.quote(part) for part in cmd)

shell_cmd = f'''
set -euo pipefail
if command -v fmriprep >/dev/null 2>&1; then
  exec {cmd_str}
elif type ml >/dev/null 2>&1; then
  ml fmriprep >/dev/null 2>&1
  exec {cmd_str}
elif type module >/dev/null 2>&1; then
  module load fmriprep >/dev/null 2>&1
  exec {cmd_str}
else
  echo "Could not find fmriprep on PATH, and module loading did not succeed." >&2
  exit 1
fi
'''

print("Command that will run:")
print(shell_cmd)


In [ ]:
if not RUN_REAL_FMRIPREP:
    raise RuntimeError(
        "Set RUN_REAL_FMRIPREP = True in the config cell before running this cell."
    )

result = subprocess.run(
    ["bash", "-lc", shell_cmd],
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"fMRIPrep failed with exit code {result.returncode}")


In [ ]:
for subject in SUBJECTS:
    subject_dir = OUTPUT_DIR / f"sub-{subject}" / "func"
    print(f"
sub-{subject}")
    if not subject_dir.exists():
        print(f"  Missing func dir: {subject_dir}")
        continue

    preproc_files = sorted(subject_dir.glob(f"*task-{TASK_ID}*echo-{ECHO_IDX}*desc-preproc_bold.nii.gz"))
    confound_files = sorted(subject_dir.glob(f"*task-{TASK_ID}*echo-{ECHO_IDX}*desc-confounds_timeseries.tsv"))
    html_report = OUTPUT_DIR / f"sub-{subject}.html"

    print("  Preprocessed BOLD files:")
    for path in preproc_files:
        print(f"    {path}")

    print("  Confounds files:")
    for path in confound_files:
        print(f"    {path}")

    print(f"  HTML report: {html_report}")


## Next step for FEAT

After this finishes, the FEAT input you want is the echo-2 preprocessed BOLD file listed in the final code cell, along with the matching echo-2 confounds TSV.

Because this is an echo-specific branch, do not point FEAT at the old `mcflirt` fallback output if your goal is to analyze the echo-2 preprocessing stream.
